# PAAC-VN · E5 GPU inference · 6 cells

Kaggle chỉ encode và xuất embeddings/app IDs/metadata để tải về local.
Không chạy smoke comparison, token-equality audit, historical comparison, clustering hoặc thống kê.
Bật **GPU + Internet**, Add Input gói mới chỉ chứa hai Parquet. Giữ nguyên FP32/batch8 và sáu cells riêng.
**Chỉ đặt AUTHOR_APPROVED_FULL=True sau khi tác giả phê duyệt chạy đủ sáu cells.**


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import os, sys, json, hashlib, subprocess, zipfile, shutil

INPUT_PATH = ""  # Để trống khi Add Input chỉ có một gói Stage02 inference.
AUTHOR_APPROVED_FULL = False
APPROVED_BY = ""
APPROVAL_REFERENCE = ""  # Tham chiếu phê duyệt thật của tác giả cho GPU inference sáu cells.

INPUT_BASE = Path("/kaggle/input")
WORK_BASE = Path("/kaggle/working")
SESSION_TAG = datetime.now(timezone.utc).strftime("kg_%Y%m%d_%H%M%S")
REPO = WORK_BASE / "PAAC-VN-E5"
INFERENCE_ID = SESSION_TAG + "_inference"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["HF_HOME"] = str(WORK_BASE / "paac_hf_cache")
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["PYTHONUNBUFFERED"] = "1"
print("Run ID:", INFERENCE_ID)


## 1. Code và input

Tự tìm ZIP hoặc thư mục đã được Kaggle giải nén, khôi phục đúng source commit và hai dataset.
Các hash/ID guards chỉ để tránh nhầm đầu vào; đánh giá embeddings thực hiện ở local.


In [ ]:
def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()

if INPUT_PATH:
    input_path = Path(INPUT_PATH)
else:
    candidates = list(INPUT_BASE.rglob("bundle_manifest.json"))
    if len(candidates) == 1:
        input_path = candidates[0].parent
    else:
        candidates = list(INPUT_BASE.rglob("paac_vn_stage02_kaggle.zip"))
        if len(candidates) != 1:
            raise RuntimeError("Hãy Add Input chứa một gói Stage02 hoặc điền INPUT_PATH cụ thể.")
        input_path = candidates[0]

if input_path.is_file():
    unpacked = WORK_BASE / "stage02_inference_input"
    if not unpacked.exists():
        with zipfile.ZipFile(input_path) as archive:
            for item in archive.infolist():
                target = (unpacked / item.filename).resolve()
                if not target.is_relative_to(unpacked.resolve()):
                    raise RuntimeError("Đường dẫn ZIP không hợp lệ.")
            archive.extractall(unpacked)
    bundle_root = unpacked
else:
    bundle_root = input_path

manifest = json.loads((bundle_root / "bundle_manifest.json").read_text())
assert manifest.get("execution_scope") == "INFERENCE_ONLY_LOCAL_VALIDATION", "Dùng gói input inference mới."
assert len(manifest["payload"]) == 2, "Input chỉ cần hai Parquet."
assert sha256(bundle_root / "repo.bundle") == manifest["git_bundle_sha256"], "Sai Git bundle hash"
if not REPO.exists():
    subprocess.run(["git", "clone", "--no-checkout", str(bundle_root / "repo.bundle"), str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", manifest["commit"]], check=True)
actual_commit = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert actual_commit == manifest["commit"], "Repo hiện tại thuộc commit khác; dùng working directory mới."
for name, digest in manifest["source_sha256"].items():
    assert sha256(REPO / name) == digest, f"Code/config khác gói: {name}"
for item in manifest["payload"]:
    source = bundle_root / "payload" / item["path"]
    assert sha256(source) == item["sha256"], f"Sai payload hash: {item['path']}"
    target = REPO / item["path"]
    target.parent.mkdir(parents=True, exist_ok=True)
    if not target.exists():
        try:
            target.symlink_to(source.resolve())
        except OSError:
            shutil.copyfile(source, target)
    assert sha256(target) == item["sha256"], f"Đầu vào hiện tại khác gói: {item['path']}"
print("Đã khôi phục", actual_commit, "| payload:", len(manifest["payload"]), "files")


## 2. Môi trường

Giữ PyTorch GPU của Kaggle, dùng dependency riêng và lưu phiên bản thực tế. Không cài PyTorch CPU hoặc thay đổi batch/precision để chạy được.


In [ ]:
base_runtime = json.loads(subprocess.check_output([
    sys.executable, "-c",
    "import torch,json; print(json.dumps({'torch':torch.__version__, 'cuda':torch.cuda.is_available(), 'gpu':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}))"
], text=True))
if not base_runtime["cuda"]:
    raise RuntimeError("Bật Settings → Accelerator → GPU, khởi động session GPU rồi chạy lại từ đầu.")
print(base_runtime)
VENV = WORK_BASE / "paac-e5-venv"
PYTHON = VENV / "bin/python"
if not PYTHON.exists():
    subprocess.run([sys.executable, "-m", "venv", "--system-site-packages", str(VENV)], check=True)
ENV_DIR = REPO / "results/e5_stage02" / (SESSION_TAG + "_launcher")
ENV_DIR.mkdir(parents=True, exist_ok=True)
requirements = REPO / "kaggle/requirements.txt"
marker = VENV / "stage02_setup.json"
setup_key = {"requirements_sha256": sha256(requirements), "base_torch": base_runtime["torch"]}
if marker.exists():
    assert json.loads(marker.read_text()) == setup_key, "Environment cũ khác requirements/torch; cần working session mới."
else:
    constraints = VENV / "torch_constraint.txt"
    constraints.write_text("torch==" + base_runtime["torch"] + "\n")
    subprocess.run([str(PYTHON), "-m", "pip", "--disable-pip-version-check", "install",
                    "-c", str(constraints), "-r", str(requirements),
                    "--log", str(ENV_DIR / "dependency_install.log")], check=True)
    marker.write_text(json.dumps(setup_key))
check = json.loads(subprocess.check_output([
    str(PYTHON), "-c",
    "import torch,json; print(json.dumps({'torch':torch.__version__, 'cuda':torch.cuda.is_available()}))"
], text=True))
assert check["cuda"] and check["torch"] == base_runtime["torch"], "PyTorch GPU đã thay đổi; dừng."
freeze = subprocess.check_output([str(PYTHON), "-m", "pip", "freeze", "--all"], text=True)
(ENV_DIR / "environment_freeze.txt").write_text(freeze)
(ENV_DIR / "base_runtime.json").write_text(json.dumps(base_runtime, indent=2))
print("GPU environment sẵn sàng:", check)


## 3. Encode sáu cells và tải ZIP

Tác giả chưa phê duyệt → bỏ qua. Đã phê duyệt → encode đủ sáu cells, không reuse Native/T512.
Mỗi cell xuất NPY, app IDs JSON, row sidecar và metadata; trạng thái PENDING_LOCAL.
Nếu chạy lỗi, ZIP giữ phần đã ghi cùng log để xử lý ở local.


In [ ]:
from IPython.display import display, FileLink
RESULTS = REPO / "results/e5_stage02"
RUN_OUTPUT = RESULTS / INFERENCE_ID

def export_embeddings():
    archive_path = WORK_BASE / "stage02_embeddings.zip"
    if archive_path.exists():
        archive_path = WORK_BASE / ("stage02_embeddings_" + SESSION_TAG + ".zip")
    with zipfile.ZipFile(archive_path, "x", zipfile.ZIP_DEFLATED, compresslevel=1) as zipped:
        for folder in (RUN_OUTPUT, ENV_DIR):
            if folder.exists():
                for path in sorted(folder.rglob("*")):
                    if path.is_file():
                        zipped.write(path, path.relative_to(REPO).as_posix())
    print("Tải về local:", archive_path, "| MB:", round(archive_path.stat().st_size / 1e6, 1))
    display(FileLink(str(archive_path)))

if not AUTHOR_APPROVED_FULL:
    print("Chưa có phê duyệt: không tải checkpoint, không chạy inference.")
else:
    assert APPROVED_BY.strip() and APPROVAL_REFERENCE.strip(), "Điền người duyệt và tham chiếu phê duyệt."
    binding = json.loads(subprocess.check_output([
        str(PYTHON), "-B", "-c",
        "import json; from pipeline.e5_stage02 import source_binding; print(json.dumps(source_binding()))"
    ], cwd=REPO, text=True))
    authorization = {
        "full_run_authorized": True,
        "approved_by": APPROVED_BY.strip(),
        "approval_reference": APPROVAL_REFERENCE.strip(),
        "full_run_id": INFERENCE_ID,
        "source_binding": binding,
        "execution_scope": "INFERENCE_ONLY_LOCAL_VALIDATION",
    }
    approval_path = ENV_DIR / "full_authorization.json"
    with approval_path.open("x") as handle:
        json.dump(authorization, handle, indent=2)
    command = [str(PYTHON), "-B", "-m", "pipeline.e5_stage02", "inference", INFERENCE_ID,
               "--full-authorization", str(approval_path), "--allow-download"]
    print("Chạy:", " ".join(command), flush=True)
    try:
        with (ENV_DIR / "inference.log").open("x") as log:
            process = subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE,
                                       stderr=subprocess.STDOUT, text=True, bufsize=1,
                                       env=os.environ.copy())
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
            returncode = process.wait()
        if returncode:
            raise RuntimeError("Inference dừng; tải ZIP/log về local. Không tự đổi cấu hình.")
        receipt = json.loads((RUN_OUTPUT / "execution_receipt.json").read_text())
        print(receipt["status"], "|", len(receipt["cells"]), "cells | validation: PENDING_LOCAL")
    finally:
        export_embeddings()


## Ở local sau khi tải về

Dùng NPY cùng app IDs/row sidecars và metadata trong ZIP để validation, so sánh lịch sử, clustering và thống kê.
INFERENCE_COMPLETE_UNVALIDATED chỉ xác nhận đã xuất xong, không thay kết quả validation.
Kaggle không đọc dữ liệu lịch sử và không thực hiện các phân tích này.
